This notebook is made to show and implement how data is downloaded via api call and also performed cleaning procedures.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys

import pandas as pd
import requests

ROOT = Path.cwd()

# Notebook notebooks/ folder ke andar ho toh parent project root hai.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Notebook ko project root ya notebooks/ se kholo.")

print("Python:", sys.executable)
print("Project root:", ROOT)

Python: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/.venv/bin/python
Project root: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment


In [2]:
COUNTRY_CODE = "NPL"
INDICATOR_CODE = "NY.GDP.PCAP.KD"

url = (
    f"https://api.worldbank.org/v2/country/"
    f"{COUNTRY_CODE}/indicator/{INDICATOR_CODE}"
)

params = {
    "source": 2,
    "date": "2010:2024",
    "format": "json",
    "per_page": 100,
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

payload = response.json()

# API kabhi HTTP 200 ke saath bhi error message bhej sakti hai.
if not (
    isinstance(payload, list)
    and len(payload) == 2
    and isinstance(payload[0], dict)
    and isinstance(payload[1], list)
):
    raise ValueError(f"Unexpected World Bank response: {payload}")

download_time = datetime.now(timezone.utc)
stamp = download_time.strftime("%Y%m%dT%H%M%S%fZ")

# Original response ka separate, timestamped copy.
raw_file = ROOT / "data" / "raw" / f"{INDICATOR_CODE}_{stamp}.json"
raw_file.write_bytes(response.content)

download_record = {
    "downloaded_at_utc": download_time.isoformat(),
    "url": response.url,
    "source_last_updated": payload[0].get("lastupdated"),
    "indicator_code": INDICATOR_CODE,
}

metadata_file = ROOT / "data" / "metadata" / f"download_{stamp}.json"
metadata_file.write_text(
    json.dumps(download_record, indent=2),
    encoding="utf-8",
)

print("Raw data saved:", raw_file.name)
print("Annual records returned:", len(payload[1]))

Raw data saved: NY.GDP.PCAP.KD_20261007T163511493221Z.json
Annual records returned: 15


In [3]:
COUNTRY_CODE = "NPL"
INDICATOR_CODE = "NY.GDP.PCAP.KD"

url = (
    f"https://api.worldbank.org/v2/country/"
    f"{COUNTRY_CODE}/indicator/{INDICATOR_CODE}"
)

params = {
    "source": 2,
    "date": "2010:2024",
    "format": "json",
    "per_page": 100,
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

payload = response.json()

# API kabhi HTTP 200 ke saath bhi error message bhej sakti hai.
if not (
    isinstance(payload, list)
    and len(payload) == 2
    and isinstance(payload[0], dict)
    and isinstance(payload[1], list)
):
    raise ValueError(f"Unexpected World Bank response: {payload}")

download_time = datetime.now(timezone.utc)
stamp = download_time.strftime("%Y%m%dT%H%M%S%fZ")

# Original response ka separate, timestamped copy.
raw_file = ROOT / "data" / "raw" / f"{INDICATOR_CODE}_{stamp}.json"
raw_file.write_bytes(response.content)

download_record = {
    "downloaded_at_utc": download_time.isoformat(),
    "url": response.url,
    "source_last_updated": payload[0].get("lastupdated"),
    "indicator_code": INDICATOR_CODE,
}

metadata_file = ROOT / "data" / "metadata" / f"download_{stamp}.json"
metadata_file.write_text(
    json.dumps(download_record, indent=2),
    encoding="utf-8",
)

print("Raw data saved:", raw_file.name)
print("Annual records returned:", len(payload[1]))

Raw data saved: NY.GDP.PCAP.KD_20261007T163650044805Z.json
Annual records returned: 15


This below cell is to convert observations data in the readable format

In [4]:
rows = [
    {
        "Year": int(observation["date"]),
        "GDP_per_capita": observation["value"],
    }
    for observation in payload[1]
]

gdp = (
    pd.DataFrame(rows)
    .set_index("Year")
    .reindex(range(2010, 2025))
    .reset_index()
)

gdp["Country"] = "Nepal"
gdp["Country_Code"] = COUNTRY_CODE

gdp = gdp[
    ["Year", "Country", "Country_Code", "GDP_per_capita"]
]

gdp.to_csv(
    ROOT / "data" / "processed" / "nepal_gdp_per_capita_2010_2024.csv",
    index=False,
)

print("Potential observations:", len(gdp))
print("Available observations:", gdp["GDP_per_capita"].notna().sum())
print("Missing observations:", gdp["GDP_per_capita"].isna().sum())

gdp

Potential observations: 15
Available observations: 15
Missing observations: 0


,Year,Country,Country_Code,GDP_per_capita
0,2010,Nepal,NPL,721.265221
1,2011,Nepal,NPL,742.617253
2,2012,Nepal,NPL,775.310578
3,2013,Nepal,NPL,801.039061
4,2014,Nepal,NPL,846.665542
5,2015,Nepal,NPL,875.543635
6,2016,Nepal,NPL,875.188944
7,2017,Nepal,NPL,951.857299
8,2018,Nepal,NPL,1021.914922
9,2019,Nepal,NPL,1077.117965


In [5]:
INDICATORS = {
    "GDP_per_capita": "NY.GDP.PCAP.KD",
    "GDP_growth": "NY.GDP.MKTP.KD.ZG",
    "Life_expectancy": "SP.DYN.LE00.IN",
    "Basic_drinking_water": "SH.H2O.BASW.ZS",
    "Secondary_school_enrollment": "SE.SEC.ENRR",
    "Electricity_access": "EG.ELC.ACCS.ZS",
    "Internet_users": "IT.NET.USER.ZS",
    "CO2_per_capita": "EN.ATM.CO2E.PC",
    "Forest_area": "AG.LND.FRST.ZS",
}

PARAMS = {
    "source": 2,
    "date": "2010:2024",
    "format": "json",
    "per_page": 100,
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

# Is batch ke original responses ek separate folder mein save honge.
RAW_DIR = ROOT / "data" / "raw" / RUN_ID
RAW_DIR.mkdir(parents=True, exist_ok=True)

nepal = pd.DataFrame({
    "Year": range(2010, 2025),
    "Country": "Nepal",
    "Country_Code": "NPL",
})

download_status = {}
download_log = {}

print("Download batch:", RUN_ID)

Download batch: 20261007T164500800228Z


In [6]:
for column, code in INDICATORS.items():
    nepal[column] = float("nan")

    url = f"https://api.worldbank.org/v2/country/NPL/indicator/{code}"
    response_file = RAW_DIR / f"{code}.json"

    try:
        response = requests.get(url, params=PARAMS, timeout=60)

        # Received response preserve karo, including API error responses.
        response_file.write_bytes(response.content)
        response.raise_for_status()

        payload = response.json()

        if not (
            isinstance(payload, list)
            and len(payload) == 2
            and isinstance(payload[0], dict)
        ):
            raise ValueError(f"Unexpected API response: {str(payload)[:300]}")

        header, observations = payload

        if int(header.get("pages", 1)) > 1:
            raise ValueError("Response has multiple pages; pagination required.")

        # A valid empty response is different from a failed download.
        if observations is None and int(header.get("total", -1)) == 0:
            observations = []

        if not isinstance(observations, list):
            raise ValueError("API did not return an observations list.")

        years = [int(row["date"]) for row in observations]

        if len(years) != len(set(years)):
            raise ValueError("Duplicate years found in the API response.")

        values = {
            int(row["date"]): row["value"]
            for row in observations
        }

        # Match values by year; unavailable values remain missing.
        nepal[column] = pd.to_numeric(
            nepal["Year"].map(values),
            errors="raise",
        )

        download_status[column] = "OK"
        download_log[column] = {
            "code": code,
            "url": response.url,
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_last_updated": header.get("lastupdated"),
            "raw_file": str(response_file.relative_to(ROOT)),
            "status": "OK",
        }

        n = int(nepal[column].notna().sum())
        print(f"{column}: {n}/15 values available")

    except (requests.RequestException, ValueError, KeyError, TypeError) as error:
        download_status[column] = "DOWNLOAD_FAILED"
        download_log[column] = {
            "code": code,
            "status": "DOWNLOAD_FAILED",
            "error": str(error),
        }

        print(f"{column}: DOWNLOAD FAILED — {error}")

GDP_per_capita: 15/15 values available
GDP_growth: 15/15 values available
Life_expectancy: 15/15 values available
Basic_drinking_water: 15/15 values available
Secondary_school_enrollment: 14/15 values available
Electricity_access: 15/15 values available
Internet_users: 15/15 values available
CO2_per_capita: DOWNLOAD FAILED — Unexpected API response: [{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]
Forest_area: 14/15 values available


To download co2 per capita from EDGAR/JRC

In [7]:
CO2_CANDIDATE_CODE = "EN.GHG.CO2.PC.CE.AR5"

candidate_url = (
    "https://api.worldbank.org/v2/country/NPL/indicator/"
    f"{CO2_CANDIDATE_CODE}"
)

candidate_response = requests.get(
    candidate_url,
    params=PARAMS,
    timeout=60,
)

candidate_time = datetime.now(timezone.utc)
candidate_stamp = candidate_time.strftime("%Y%m%dT%H%M%S%fZ")

# Original response preserve karo.
candidate_raw = (
    ROOT / "data" / "raw"
    / f"{CO2_CANDIDATE_CODE}_{candidate_stamp}.json"
)
candidate_raw.write_bytes(candidate_response.content)

candidate_response.raise_for_status()
candidate_payload = candidate_response.json()

if not (
    isinstance(candidate_payload, list)
    and len(candidate_payload) == 2
    and isinstance(candidate_payload[0], dict)
    and isinstance(candidate_payload[1], list)
):
    raise ValueError(f"Unexpected API response: {candidate_payload}")

co2_candidate = pd.DataFrame(
    [
        {
            "Year": int(row["date"]),
            "CO2_per_capita_excl_LULUCF": row["value"],
        }
        for row in candidate_payload[1]
    ],
    columns=["Year", "CO2_per_capita_excl_LULUCF"],
)

co2_candidate = (
    co2_candidate
    .set_index("Year")
    .reindex(range(2010, 2025))
    .reset_index()
)

candidate_record = {
    "assignment_code": "EN.ATM.CO2E.PC",
    "candidate_code": CO2_CANDIDATE_CODE,
    "unit": "t CO2e/capita",
    "downloaded_at_utc": candidate_time.isoformat(),
    "url": candidate_response.url,
    "source_last_updated": candidate_payload[0].get("lastupdated"),
    "role": "Candidate alternative series",
}

(ROOT / "data" / "metadata" / f"co2_candidate_{candidate_stamp}.json").write_text(
    json.dumps(candidate_record, indent=2),
    encoding="utf-8",
)

co2_candidate.to_csv(
    ROOT / "data" / "processed" / f"nepal_co2_candidate_{candidate_stamp}.csv",
    index=False,
)

print(
    "Available CO2 observations:",
    co2_candidate["CO2_per_capita_excl_LULUCF"].notna().sum(),
    "/ 15",
)

display(co2_candidate)

Available CO2 observations: 15 / 15


,Year,CO2_per_capita_excl_LULUCF
0,2010,0.175006
1,2011,0.195187
2,2012,0.226080
3,2013,0.228210
4,2014,0.265695
5,2015,0.263161
6,2016,0.385041
7,2017,0.471232
8,2018,0.531555
9,2019,0.473751


Saving coverage tables and saving files from there..

In [8]:
coverage_rows = []

for column, code in INDICATORS.items():
    success = download_status[column] == "OK"

    n = int(nepal[column].notna().sum()) if success else None

    missing_years = (
        nepal.loc[nepal[column].isna(), "Year"].tolist()
        if success else []
    )

    coverage_rows.append({
        "Indicator": column,
        "Code": code,
        "N": n,
        "Missing": 15 - n if success else None,
        "Missing_years": (
            ", ".join(map(str, missing_years))
            if success else "Not assessed: download failed"
        ),
        "Status": download_status[column],
    })

coverage = pd.DataFrame(coverage_rows)
coverage[["N", "Missing"]] = coverage[["N", "Missing"]].astype("Int64")

failed = [
    column for column, status in download_status.items()
    if status != "OK"
]

filename = (
    "nepal_wdi_2010_2024_provisional.csv"
    if failed else "nepal_wdi_2010_2024.csv"
)

nepal.to_csv(ROOT / "data" / "processed" / filename, index=False)

coverage.to_csv(
    ROOT / "results" / "tables" / "data_coverage.csv",
    index=False,
)

(ROOT / "data" / "metadata" / f"download_{RUN_ID}.json").write_text(
    json.dumps(download_log, indent=2),
    encoding="utf-8",
)

display(coverage)
display(nepal.head())

print("Dataset saved:", filename)

if failed:
    print("Downloads requiring investigation:", failed)
else:
    print("All nine downloads completed; coverage is ready for review.")

,Indicator,Code,N,Missing,Missing_years,Status
0,GDP_per_capita,NY.GDP.PCAP.KD,15,0,,OK
1,GDP_growth,NY.GDP.MKTP.KD.ZG,15,0,,OK
2,Life_expectancy,SP.DYN.LE00.IN,15,0,,OK
3,Basic_drinking_water,SH.H2O.BASW.ZS,15,0,,OK
4,Secondary_school_enrollment,SE.SEC.ENRR,14,1,2018,OK
5,Electricity_access,EG.ELC.ACCS.ZS,15,0,,OK
6,Internet_users,IT.NET.USER.ZS,15,0,,OK
7,CO2_per_capita,EN.ATM.CO2E.PC,<NA>,<NA>,Not assessed: download failed,DOWNLOAD_FAILED
8,Forest_area,AG.LND.FRST.ZS,14,1,2024,OK


,Year,Country,Country_Code,GDP_per_capita,GDP_growth,Life_expectancy,Basic_drinking_water,Secondary_school_enrollment,Electricity_access,Internet_users,CO2_per_capita,Forest_area
0,2010,Nepal,NPL,721.265221,4.816415,66.772,85.761759,55.589291,68.6,7.9300,NaN,41.590722
1,2011,Nepal,NPL,742.617253,3.421809,67.123,86.359902,58.993511,67.3,9.0000,NaN,41.590722
2,2012,Nepal,NPL,775.310578,4.670142,67.364,86.953284,63.141479,75.2,11.1493,NaN,41.590722
3,2013,Nepal,NPL,801.039061,3.525153,67.598,87.541580,65.493530,77.4,13.3000,NaN,41.590722
4,2014,Nepal,NPL,846.665542,6.011483,67.812,88.124806,67.531830,84.9,15.4400,NaN,41.590722


Dataset saved: nepal_wdi_2010_2024_provisional.csv
Downloads requiring investigation: ['CO2_per_capita']


Merging the previous cell

In [9]:
CO2_COLUMN = "CO2_per_capita_excl_LULUCF"
CO2_CODE = "EN.GHG.CO2.PC.CE.AR5"

# Working dataset mein empty legacy CO2 column hata kar
# downloaded candidate series ko Year se match karo.
nepal_working = (
    nepal.drop(columns=["CO2_per_capita"])
    .merge(
        co2_candidate[["Year", CO2_COLUMN]],
        on="Year",
        how="left",
        validate="one_to_one",
    )
)

# Working dataset ki coverage mein actual code record karo.
coverage_working = coverage.copy()
coverage_working = coverage_working.rename(
    columns={"Code": "Code_used"}
)
coverage_working["Code_assigned"] = coverage_working["Code_used"]

co2_row = coverage_working["Indicator"].eq("CO2_per_capita")

n = int(nepal_working[CO2_COLUMN].notna().sum())
missing_years = nepal_working.loc[
    nepal_working[CO2_COLUMN].isna(), "Year"
].tolist()

coverage_working.loc[co2_row, "Indicator"] = CO2_COLUMN
coverage_working.loc[co2_row, "Code_used"] = CO2_CODE
coverage_working.loc[co2_row, "N"] = n
coverage_working.loc[co2_row, "Missing"] = len(nepal_working) - n
coverage_working.loc[co2_row, "Missing_years"] = ", ".join(
    map(str, missing_years)
)
coverage_working.loc[co2_row, "Status"] = "OK_ALTERNATIVE"

# Combined working dataset aur updated coverage save karo.
nepal_working.to_csv(
    ROOT / "data" / "processed" / "nepal_wdi_2010_2024_working.csv",
    index=False,
)

coverage_working.to_csv(
    ROOT / "results" / "tables" / "working_data_coverage.csv",
    index=False,
)

display(coverage_working)
display(nepal_working[["Year", "GDP_per_capita", CO2_COLUMN]])

print("Combined working dataset saved.")

,Indicator,Code_used,N,Missing,Missing_years,Status,Code_assigned
0,GDP_per_capita,NY.GDP.PCAP.KD,15,0,,OK,NY.GDP.PCAP.KD
1,GDP_growth,NY.GDP.MKTP.KD.ZG,15,0,,OK,NY.GDP.MKTP.KD.ZG
2,Life_expectancy,SP.DYN.LE00.IN,15,0,,OK,SP.DYN.LE00.IN
3,Basic_drinking_water,SH.H2O.BASW.ZS,15,0,,OK,SH.H2O.BASW.ZS
4,Secondary_school_enrollment,SE.SEC.ENRR,14,1,2018,OK,SE.SEC.ENRR
5,Electricity_access,EG.ELC.ACCS.ZS,15,0,,OK,EG.ELC.ACCS.ZS
6,Internet_users,IT.NET.USER.ZS,15,0,,OK,IT.NET.USER.ZS
7,CO2_per_capita_excl_LULUCF,EN.GHG.CO2.PC.CE.AR5,15,0,,OK_ALTERNATIVE,EN.ATM.CO2E.PC
8,Forest_area,AG.LND.FRST.ZS,14,1,2024,OK,AG.LND.FRST.ZS


,Year,GDP_per_capita,CO2_per_capita_excl_LULUCF
0,2010,721.265221,0.175006
1,2011,742.617253,0.195187
2,2012,775.310578,0.226080
3,2013,801.039061,0.228210
4,2014,846.665542,0.265695
5,2015,875.543635,0.263161
6,2016,875.188944,0.385041
7,2017,951.857299,0.471232
8,2018,1021.914922,0.531555
9,2019,1077.117965,0.473751


Combined working dataset saved.


In [10]:
# Indicators ko Year–Indicator–Value format mein dekho.
missing_details = (
    nepal_working
    .drop(columns=["Country", "Country_Code"])
    .melt(
        id_vars="Year",
        var_name="Indicator",
        value_name="Value",
    )
)

# Sirf missing observations dikhao.
missing_details = missing_details.loc[
    missing_details["Value"].isna()
].reset_index(drop=True)

display(missing_details)

,Year,Indicator,Value
0,2018,Secondary_school_enrollment,NaN
1,2024,Forest_area,NaN


In [11]:
display(
    nepal_working.loc[
        nepal_working.isna().any(axis=1)
    ]
)

,Year,Country,Country_Code,GDP_per_capita,GDP_growth,Life_expectancy,Basic_drinking_water,Secondary_school_enrollment,Electricity_access,Internet_users,Forest_area,CO2_per_capita_excl_LULUCF
8,2018,Nepal,NPL,1021.914922,7.622376,69.036,90.403234,NaN,93.9,28.600000,41.590722,0.531555
14,2024,Nepal,NPL,1179.954993,3.677875,70.636,93.640125,90.387846,97.9,46.321899,NaN,0.535964


TO see that which is the file source of the data and what exactly are the missing datas...

In [12]:
DATA_FILE = (
    ROOT / "data" / "processed"
    / "nepal_wdi_2010_2024_working.csv"
)

nepal_working = pd.read_csv(DATA_FILE)

print("Loaded from:", DATA_FILE.resolve())
print("Rows, columns:", nepal_working.shape)

missing_details = (
    nepal_working
    .drop(columns=["Country", "Country_Code"])
    .melt(id_vars="Year", var_name="Indicator", value_name="Value")
)

missing_details = missing_details.loc[
    missing_details["Value"].isna()
].reset_index(drop=True)

display(missing_details)

Loaded from: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/data/processed/nepal_wdi_2010_2024_working.csv
Rows, columns: (15, 12)


,Year,Indicator,Value
0,2018,Secondary_school_enrollment,NaN
1,2024,Forest_area,NaN


For school enrollment: And as Source UNESCO UIS hai. Tumhare snapshot mein gap hai; iska specific reason metadata mein nahi mila.
aur for forest area: WDI metadata ka reference period 1990–2023 hai. Yeh 2024 ki missing value ke saath consistent hai.

In [13]:
targets = [
    ("Secondary_school_enrollment", "SE.SEC.ENRR", 2018),
    ("Forest_area", "AG.LND.FRST.ZS", 2024),
]

check_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
check_dir = ROOT / "data" / "raw" / f"missing_recheck_{check_id}"
check_dir.mkdir(parents=True, exist_ok=True)

check_rows = []

for column, code, year in targets:
    print(f"\nInvestigating: {column}, {year}")

    # Missing year aur uske neighboring observations.
    display(
        nepal_working.loc[
            nepal_working["Year"].between(year - 1, year + 1),
            ["Year", column],
        ]
    )

    response = requests.get(
        f"https://api.worldbank.org/v2/country/NPL/indicator/{code}",
        params={
            "source": 2,
            "date": str(year),
            "format": "json",
            "per_page": 100,
        },
        timeout=60,
    )

    # Error response bhi preserve hogi.
    raw_file = check_dir / f"{code}_{year}.json"
    raw_file.write_bytes(response.content)

    response.raise_for_status()
    payload = response.json()

    if not (
        isinstance(payload, list)
        and len(payload) == 2
        and isinstance(payload[0], dict)
        and isinstance(payload[1], (list, type(None)))
    ):
        raise ValueError(f"Unexpected API response: {payload}")

    records = payload[1] or []

    if len(records) > 1 or any(
        int(row["date"]) != year
        or row["countryiso3code"] != "NPL"
        or row["indicator"]["id"] != code
        for row in records
    ):
        raise ValueError("Response does not match the requested observation.")

    value = records[0]["value"] if records else None

    check_rows.append({
        "Indicator": column,
        "Year": year,
        "Code": code,
        "WDI_value": value,
        "Status": (
            "VALUE_AVAILABLE" if value is not None
            else "NO_VALUE_RETURNED"
        ),
        "URL": response.url,
        "Source_last_updated": payload[0].get("lastupdated"),
        "Raw_file": str(raw_file.relative_to(ROOT)),
    })

missing_recheck = pd.DataFrame(check_rows)

missing_recheck.to_csv(
    ROOT / "results" / "tables" / f"missing_recheck_{check_id}.csv",
    index=False,
)

display(missing_recheck)


Investigating: Secondary_school_enrollment, 2018


,Year,Secondary_school_enrollment
7,2017,74.411331
8,2018,NaN
9,2019,80.280052



Investigating: Forest_area, 2024


,Year,Forest_area
13,2023,41.590722
14,2024,NaN


,Indicator,Year,Code,WDI_value,Status,URL,Source_last_updated,Raw_file
0,Secondary_school_enrollment,2018,SE.SEC.ENRR,None,NO_VALUE_RETURNED,https://api.worldbank.org/v2/country/NPL/indic...,2026-07-13,data/raw/missing_recheck_20261008T130454594601...
1,Forest_area,2024,AG.LND.FRST.ZS,None,NO_VALUE_RETURNED,https://api.worldbank.org/v2/country/NPL/indic...,2026-07-13,data/raw/missing_recheck_20261008T130454594601...


As per the assignment, it is clearly mentioned & is said that.. Missing observations must remain missing and must be reported... 
Therefore in the cleaning part, i have not made any changes or so to the data and have let it remain Nan only... 

In [14]:
columns = ["Secondary_school_enrollment", "Forest_area"]

missing_summary = pd.DataFrame({
    "Available_N": nepal_working[columns].count(),
    "Missing_N": nepal_working[columns].isna().sum(),
    "Observed_mean": nepal_working[columns].mean(),
})

display(missing_summary)

,Available_N,Missing_N,Observed_mean
Secondary_school_enrollment,14,1,74.245800
Forest_area,14,1,41.590722


Now: for the data Quality checks..


In [15]:
import numpy as np

# Saved working dataset ko explicitly load karo.
DATA_FILE = (
    ROOT / "data" / "processed"
    / "nepal_wdi_2010_2024_working.csv"
)
nepal_working = pd.read_csv(DATA_FILE)

indicator_columns = [
    "GDP_per_capita",
    "GDP_growth",
    "Life_expectancy",
    "Basic_drinking_water",
    "Secondary_school_enrollment",
    "Electricity_access",
    "Internet_users",
    "CO2_per_capita_excl_LULUCF",
    "Forest_area",
]

# Required columns available honi chahiye.
required = ["Year", "Country", "Country_Code"] + indicator_columns
missing_columns = [c for c in required if c not in nepal_working.columns]

if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

numeric_ok = all(
    pd.api.types.is_numeric_dtype(nepal_working[c])
    for c in indicator_columns
)

checks = {
    "Exactly 15 annual rows": len(nepal_working) == 15,
    "Years are 2010–2024": (
        sorted(nepal_working["Year"].tolist())
        == list(range(2010, 2025))
    ),
    "No duplicate years": not nepal_working["Year"].duplicated().any(),
    "Country is Nepal": nepal_working["Country"].eq("Nepal").all(),
    "Country code is NPL": nepal_working["Country_Code"].eq("NPL").all(),
    "All indicators are numeric": numeric_ok,
}

if numeric_ok:
    checks["Observed values are finite"] = all(
        np.isfinite(nepal_working[c].dropna()).all()
        for c in indicator_columns
    )

    # Population/land shares: valid range 0–100.
    for column in [
        "Basic_drinking_water",
        "Electricity_access",
        "Internet_users",
        "Forest_area",
    ]:
        values = nepal_working[column].dropna()
        checks[f"{column}: within 0–100"] = values.between(0, 100).all()

    # Gross enrollment can exceed 100; check only its lower bound.
    for column in [
        "GDP_per_capita",
        "Secondary_school_enrollment",
        "CO2_per_capita_excl_LULUCF",
    ]:
        checks[f"{column}: non-negative"] = (
            nepal_working[column].dropna().ge(0).all()
        )

    checks["Life expectancy is positive"] = (
        nepal_working["Life_expectancy"].dropna().gt(0).all()
    )

quality_checks = pd.DataFrame({
    "Check": list(checks),
    "Passed": [bool(value) for value in checks.values()],
})

quality_checks["Source_file"] = str(DATA_FILE.relative_to(ROOT))
quality_checks["Checked_at_UTC"] = datetime.now(timezone.utc).isoformat()

output_file = ROOT / "results" / "tables" / "data_quality_checks.csv"
quality_checks.to_csv(output_file, index=False)

print("Dataset:", DATA_FILE.resolve())
display(quality_checks[["Check", "Passed"]])
print("Checks saved:", output_file.resolve())

if not quality_checks["Passed"].all():
    raise ValueError("A quality check failed; review it before analysis.")

Dataset: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/data/processed/nepal_wdi_2010_2024_working.csv


,Check,Passed
0,Exactly 15 annual rows,True
1,Years are 2010–2024,True
2,No duplicate years,True
3,Country is Nepal,True
4,Country code is NPL,True
5,All indicators are numeric,True
6,Observed values are finite,True
7,Basic_drinking_water: within 0–100,True
8,Electricity_access: within 0–100,True
9,Internet_users: within 0–100,True


Checks saved: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/results/tables/data_quality_checks.csv


In [16]:
# Working dataset mein jo series actually use hui hain.
VARIABLE_SPECS = {
    "GDP_per_capita": (
        "NY.GDP.PCAP.KD", "constant 2015 US$ per person"
    ),
    "GDP_growth": (
        "NY.GDP.MKTP.KD.ZG", "annual %"
    ),
    "Life_expectancy": (
        "SP.DYN.LE00.IN", "years"
    ),
    "Basic_drinking_water": (
        "SH.H2O.BASW.ZS", "% of population"
    ),
    "Secondary_school_enrollment": (
        "SE.SEC.ENRR", "% gross"
    ),
    "Electricity_access": (
        "EG.ELC.ACCS.ZS", "% of population"
    ),
    "Internet_users": (
        "IT.NET.USER.ZS", "% of population"
    ),
    "CO2_per_capita_excl_LULUCF": (
        "EN.GHG.CO2.PC.CE.AR5", "t CO2e/capita"
    ),
    "Forest_area": (
        "AG.LND.FRST.ZS", "% of land area"
    ),
}

METADATA_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
METADATA_RAW_DIR = ROOT / "data" / "raw" / f"metadata_{METADATA_ID}"
METADATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
dictionary_rows = []

for column, (code, fallback_unit) in VARIABLE_SPECS.items():
    is_alternative = column == "CO2_per_capita_excl_LULUCF"

    # Observation download date aur metadata fetch date alag hain.
    observation_log = (
        globals().get("candidate_record", {})
        if is_alternative
        else globals().get("download_log", {}).get(column, {})
    )

    missing_years = nepal_working.loc[
        nepal_working[column].isna(), "Year"
    ].tolist()

    entry = {
        "Indicator": column,
        "Code_assigned": "EN.ATM.CO2E.PC" if is_alternative else code,
        "Code_used": code,
        "Series_role": "Alternative" if is_alternative else "Assigned",
        "WDI_name": "",
        "Definition_WDI": "",
        "Unit": fallback_unit,
        "Unit_API_raw": "",
        "Unit_reference": (
            "World Bank current CO2 indicator page"
            if is_alternative
            else "Assignment indicator table"
        ),
        "Data_provider": "World Bank WDI",
        "Source_organization_WDI": "",
        "Variable_type": "Quantitative; continuous measure",
        "Frequency": "Annual",
        "Analysis_period": "2010–2024",
        "N": int(nepal_working[column].notna().sum()),
        "Missing": int(nepal_working[column].isna().sum()),
        "Missing_years": ", ".join(map(str, missing_years)),
        "Observation_downloaded_at_utc": (
            observation_log.get("downloaded_at_utc")
            or observation_log.get("downloaded_at")
            or ""
        ),
        "Metadata_downloaded_at_utc": "",
        "Metadata_URL": "",
        "Metadata_raw_file": "",
        "Metadata_status": "NOT_REQUESTED",
        "Metadata_error": "",
    }

    try:
        response = requests.get(
            f"https://api.worldbank.org/v2/indicator/{code}",
            params={"source": 2, "format": "json", "per_page": 100},
            timeout=60,
        )

        raw_file = METADATA_RAW_DIR / f"{code}.json"
        raw_file.write_bytes(response.content)

        entry["Metadata_URL"] = response.url
        entry["Metadata_raw_file"] = str(raw_file.relative_to(ROOT))
        entry["Metadata_downloaded_at_utc"] = (
            datetime.now(timezone.utc).isoformat()
        )

        response.raise_for_status()
        payload = response.json()

        if not (
            isinstance(payload, list)
            and len(payload) == 2
            and isinstance(payload[0], dict)
            and isinstance(payload[1], list)
        ):
            raise ValueError(f"Unexpected metadata response: {payload}")

        if int(payload[0].get("pages", 1)) != 1:
            raise ValueError("Metadata response requires pagination.")

        matches = [
            item for item in payload[1]
            if item["id"] == code
            and str(item["source"]["id"]) == "2"
        ]

        if len(matches) != 1:
            raise ValueError("Expected one matching WDI metadata record.")

        metadata = matches[0]

        entry["WDI_name"] = metadata.get("name") or ""
        entry["Definition_WDI"] = metadata.get("sourceNote") or ""
        entry["Source_organization_WDI"] = (
            metadata.get("sourceOrganization") or ""
        )
        entry["Unit_API_raw"] = metadata.get("unit") or ""

        if entry["Unit_API_raw"].strip():
            entry["Unit"] = entry["Unit_API_raw"].strip()
            entry["Unit_reference"] = "World Bank metadata API"

        required_fields = [
            "WDI_name", "Definition_WDI", "Source_organization_WDI"
        ]
        entry["Metadata_status"] = (
            "OK"
            if all(entry[field].strip() for field in required_fields)
            else "REVIEW_REQUIRED"
        )

    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        entry["Metadata_status"] = "DOWNLOAD_FAILED"
        entry["Metadata_error"] = str(exc)

    dictionary_rows.append(entry)
    print(column, "→", entry["Metadata_status"])

variable_dictionary = pd.DataFrame(dictionary_rows)

needs_review = (
    variable_dictionary["Metadata_status"].ne("OK").any()
    or variable_dictionary["Observation_downloaded_at_utc"].eq("").any()
)

dictionary_file = ROOT / "data" / "metadata" / (
    "variable_dictionary_provisional.csv"
    if needs_review else "variable_dictionary.csv"
)

variable_dictionary.to_csv(dictionary_file, index=False)

display(variable_dictionary[[
    "Indicator", "Code_used", "Unit", "N",
    "Missing", "Missing_years", "Metadata_status",
]])

print("Dictionary saved:", dictionary_file.resolve())

if needs_review:
    print("Review metadata failures or blank observation download dates.")
else:
    print("All nine dictionary entries are documented.")

GDP_per_capita → OK
GDP_growth → OK
Life_expectancy → OK
Basic_drinking_water → OK
Secondary_school_enrollment → OK
Electricity_access → OK
Internet_users → OK
CO2_per_capita_excl_LULUCF → OK
Forest_area → OK


,Indicator,Code_used,Unit,N,Missing,Missing_years,Metadata_status
0,GDP_per_capita,NY.GDP.PCAP.KD,constant 2015 US$ per person,15,0,,OK
1,GDP_growth,NY.GDP.MKTP.KD.ZG,annual %,15,0,,OK
2,Life_expectancy,SP.DYN.LE00.IN,years,15,0,,OK
3,Basic_drinking_water,SH.H2O.BASW.ZS,% of population,15,0,,OK
4,Secondary_school_enrollment,SE.SEC.ENRR,% gross,14,1,2018,OK
5,Electricity_access,EG.ELC.ACCS.ZS,% of population,15,0,,OK
6,Internet_users,IT.NET.USER.ZS,% of population,15,0,,OK
7,CO2_per_capita_excl_LULUCF,EN.GHG.CO2.PC.CE.AR5,t CO2e/capita,15,0,,OK
8,Forest_area,AG.LND.FRST.ZS,% of land area,14,1,2024,OK


Dictionary saved: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/data/metadata/variable_dictionary.csv
All nine dictionary entries are documented.


In [18]:
notes_path = ROOT / "data" / "metadata" / "data_quality_notes.md"

notes_path.write_text(
    """# Nepal: Data quality notes

- Country: Nepal (NPL). Analysis period: 2010–2024.
- Working dataset: data/processed/nepal_wdi_2010_2024_working.csv.
- All 15 implemented structural and plausibility checks passed.
- Secondary school enrollment is missing for 2018.
- Forest area is missing for 2024.
- Exact-year API rechecks returned no value for both missing observations.
- Missing observations remain missing; no interpolation or imputation was applied.
- Each calculation uses the available observations for its indicator.
- The assigned CO2 code EN.ATM.CO2E.PC failed to download.
- The working dataset uses EN.GHG.CO2.PC.CE.AR5 as a clearly labelled alternative:
  CO2 emissions excluding LULUCF per capita, measured in t CO2e/capita.
- This alternative is downloaded through World Bank WDI; its metadata identifies
  EDGAR/JRC–European Commission and IEA as underlying sources.
- Instructor acceptance of the CO2 substitution remains to be confirmed.
- Raw responses, download records and variable definitions are preserved.
""",
    encoding="utf-8",
)

print("Saved:", notes_path.resolve())

Saved: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/data/metadata/data_quality_notes.md
